In [ ]:
# ============================================================
# HIGH ACCURACY Interactive Spam Detector
# ============================================================

import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# ------------------------------------------------------------
# 1. Load Dataset
# ------------------------------------------------------------
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

print("Dataset size:", df.shape)
print(df['label'].value_counts())

# ------------------------------------------------------------
# 2. Advanced Text Cleaning
# ------------------------------------------------------------
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    # Remove URLs, emails, numbers, punctuation
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize + Lemmatize + Remove stopwords
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]
    return ' '.join(tokens)

df['cleaned'] = df['message'].apply(clean_text)
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# ------------------------------------------------------------
# 3. Train-Test Split
# ------------------------------------------------------------
X = df['cleaned']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ------------------------------------------------------------
# 4. TF-IDF + Models
# ------------------------------------------------------------
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),          # unigrams + bigrams
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

# Models to try
models = {
    "MultinomialNB": MultinomialNB(alpha=0.1),
    "LogisticRegression": LogisticRegression(
        C=10,
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ),
    "LinearSVC": LinearSVC(
        C=1.0,
        class_weight='balanced',
        random_state=42,
        dual=False
    )
}

print("\n" + "="*60)
print("Training & Evaluating Models...")
print("="*60)

best_model = None
best_score = 0
best_name = ""

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)

    # Cross-validation for reliability
    cv_scores = cross_val_score(model, X_train_tfidf, y_train, cv=5, scoring='accuracy')

    print(f"\n{name}")
    print(f"  Test Accuracy : {acc*100:.2f}%")
    print(f"  F1-Score      : {f1:.4f}")
    print(f"  CV Accuracy   : {cv_scores.mean()*100:.2f}% (±{cv_scores.std()*100:.2f})")

    if acc > best_score:
        best_score = acc
        best_model = model
        best_name = name

print("\n" + "="*60)
print(f"Best Model Selected → {best_name}  |  Accuracy: {best_score*100:.2f}%")
print("="*60)

# For LinearSVC we need calibration to get probabilities
if best_name == "LinearSVC":
    best_model = CalibratedClassifierCV(best_model, cv=3)
    best_model.fit(X_train_tfidf, y_train)

# ------------------------------------------------------------
# 5. Prediction Function
# ------------------------------------------------------------
def predict_message(message):
    cleaned = clean_text(message)
    vector = tfidf.transform([cleaned])

    prediction = best_model.predict(vector)[0]
    proba = best_model.predict_proba(vector)[0]

    if prediction == 1:
        result = "SPAM"
        confidence = proba[1] * 100
    else:
        result = "NOT SPAM (Ham)"
        confidence = proba[0] * 100

    return result, confidence

# ------------------------------------------------------------
# 6. Interactive User Interface
# ------------------------------------------------------------
print("\n" + "="*60)
print("          HIGH ACCURACY SPAM DETECTOR")
print("="*60)
print("Type any email/SMS message and press Enter.")
print("Type 'exit' to quit.\n")

while True:
    user_input = input("Enter message → ").strip()

    if user_input.lower() in ['exit', 'quit', 'q']:
        print("\nThank you! Exiting...")
        break

    if len(user_input) < 3:
        print("Please enter a longer message.\n")
        continue

    result, confidence = predict_message(user_input)

    print("\n" + "-"*60)
    print(f"Message     : {user_input}")
    print(f"Prediction  : {result}")
    print(f"Confidence  : {confidence:.2f}%")
    print("-"*60 + "\n")

Dataset size: (5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64

Training & Evaluating Models...

MultinomialNB
  Test Accuracy : 97.67%
  F1-Score      : 0.9078
  CV Accuracy   : 97.87% (±0.16)

LogisticRegression
  Test Accuracy : 97.94%
  F1-Score      : 0.9226
  CV Accuracy   : 98.00% (±0.43)

LinearSVC
  Test Accuracy : 98.03%
  F1-Score      : 0.9262
  CV Accuracy   : 98.21% (±0.31)

Best Model Selected → LinearSVC  |  Accuracy: 98.03%

          HIGH ACCURACY SPAM DETECTOR
Type any email/SMS message and press Enter.
Type 'exit' to quit.

Enter message → 
Please enter a longer message.

Enter message → pay 10000 to get job

------------------------------------------------------------
Message     : pay 10000 to get job
Prediction  : NOT SPAM (Ham)
Confidence  : 99.61%
------------------------------------------------------------

Enter message → you won prize ! please claim it

------------------------------------------------------------
Message     : you won priz